# NACLIP with the VS Code Colab extension

Open this notebook in VS Code and select **Select Kernel → Colab → your existing GPU server** (or **New Colab Server** when needed). Complete Google sign-in yourself. All cells execute on Colab; the ordinary VS Code terminal stays local.

The first cell clones **https://github.com/Chowkey/NACLIP.git**, branch `main`, into `/content/NACLIP` on a fresh runtime. Only committed and pushed code is available through GitHub; local edits and old uploaded ZIPs are not loaded automatically. An existing checkout is kept by default. Set `UPDATE_SOURCE=True` to fetch and fast-forward a clean checkout after active jobs finish. An existing uploaded folder without `.git` is preserved: choose a new `PROJECT` path to clone alongside it.

The setup and smoke cells use the same isolated environment as the CLI. Optional evaluation and dataset preparation remain disabled until explicitly enabled. Datasets and results live in mounted Drive. NACLIP is training-free.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

def run_remote(command, **kwargs):
    environment = os.environ.copy()
    environment['MPLBACKEND'] = 'Agg'
    kwargs.setdefault('env', environment)
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, **kwargs)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, command)

PROJECT = Path('/content/NACLIP')
PYTHON = Path('/content/naclip-env/bin/python')
REPO_URL = 'https://github.com/Chowkey/NACLIP.git'
REPO_BRANCH = 'main'
UPDATE_SOURCE = False  # Enable only after active jobs have finished.

if not Path('/content').is_dir():
    raise RuntimeError('Select a Colab kernel before running these cells.')
source_started = time.perf_counter()
if not PROJECT.exists():
    run_remote(['git', 'clone', '--depth', '1', '--single-branch',
                '--branch', REPO_BRANCH, REPO_URL, str(PROJECT)])
elif not (PROJECT / '.git').is_dir():
    raise RuntimeError(
        f'{PROJECT} already exists without Git metadata (possibly uploaded source). '
        'Choose a new PROJECT path to clone alongside it. Existing files were preserved.')
else:
    origin = subprocess.check_output(
        ['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if origin.removesuffix('.git').rstrip('/') != REPO_URL.removesuffix('.git').rstrip('/'):
        raise RuntimeError('Existing checkout has a different origin. Choose another PROJECT path.')
    if UPDATE_SOURCE:
        if subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT, text=True).strip():
            raise RuntimeError('Checkout has local changes; preserve them before updating source.')
        branch = subprocess.check_output(
            ['git', 'branch', '--show-current'], cwd=PROJECT, text=True).strip()
        if branch != REPO_BRANCH:
            raise RuntimeError(f'Existing branch is {branch!r}; expected {REPO_BRANCH!r}.')
        run_remote(['git', 'fetch', 'origin', REPO_BRANCH], cwd=PROJECT)
        run_remote(['git', 'merge', '--ff-only', 'FETCH_HEAD'], cwd=PROJECT)
    else:
        print('Keeping existing checkout. Set UPDATE_SOURCE=True to fetch pushed changes.')
print(f'Source preparation: {time.perf_counter() - source_started:.2f} seconds')
run_remote(['git', 'log', '-1', '--format=Commit: %H%nSubject: %s'], cwd=PROJECT)
required = ['tools/colab_setup.py', 'tools/colab_smoke.py', 'tools/run_baseline.py',
            'tools/prepare_datasets.py', 'tools/colab_eval.py', 'naclip.py']
missing = [name for name in required if not (PROJECT / name).is_file()]
if missing:
    raise RuntimeError(
        'Repository checkout is missing: ' + ', '.join(missing) +
        '. Commit and push the required local scripts, then update the checkout.')
run_remote(['nvidia-smi'])
print('Remote project:', PROJECT)


## Install isolated dependencies
Run once per new runtime. Downloads can take several minutes. The final import check must succeed before continuing.

In [ ]:
run_remote([sys.executable, '-u', str(PROJECT / 'tools/colab_setup.py')], cwd=PROJECT)


## GPU inference check
Downloads the pretrained CLIP weights on first use and checks finite output on a synthetic image. This does not measure segmentation quality.

In [ ]:
run_remote([str(PYTHON), '-u', 'tools/colab_smoke.py'], cwd=PROJECT)


## Optional: mount Drive
Run only if your data/results live on Drive. Complete Google authorization yourself.

In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


## Optional: upstream dataset evaluation

This is the simpler upstream evaluation entry point. For ADE20K/PC59 baseline runs with saved predictions, timing, and `summary.json`, use the baseline cells below instead. Set paths to an already prepared dataset and a persistent results folder. VOC21 expects JPEGImages, SegmentationClass, and ImageSets/Segmentation/val.txt under DATA_ROOT. Local computer paths are unavailable here.


In [ ]:
RUN_EVALUATION = False
CONFIG = 'configs/cfg_voc21.py'
DATA_ROOT = '/content/drive/MyDrive/datasets/VOCdevkit/VOC2012'
WORK_DIR = '/content/drive/MyDrive/NACLIP-results/voc21'
if RUN_EVALUATION:
    if WORK_DIR.startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Mount Drive first to persist evaluation results.')
    run_remote([str(PYTHON), '-u', 'tools/colab_eval.py', '--config', CONFIG,
                    '--data-root', DATA_ROOT, '--work-dir', WORK_DIR], cwd=PROJECT)
else:
    print('Evaluation disabled. Set dataset paths and RUN_EVALUATION=True when ready.')


## Optional: prepare ADE20K and PC59

Skip this if the datasets are already prepared in Drive. Set `PREPARE_DATASETS=True` only when preparation is needed, after mounting Drive. This follows the proposal dataset notebook and runs on the Colab notebook Python without model dependencies.


In [ ]:
PREPARE_DATASETS = False
if PREPARE_DATASETS:
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Mount Google Drive first.')
    run_remote([sys.executable, '-u', 'tools/prepare_datasets.py',
                '--data-root', '/content/drive/MyDrive/datasets'], cwd=PROJECT)
else:
    print('Dataset preparation disabled; reuse the prepared Drive datasets.')


## Optional: fixed subset or full validation baseline

Choose `DATASET='ade20k'` or `'pc59'` and `STAGE='subset'` or `'full'`. Subset runs use 32 fixed images with seed 42 and save raw prediction PNGs plus comparison JPEGs (input / ground truth / prediction). Full runs evaluate all 2,000 ADE20K or 5,105 PC59 validation images. Both save config, package versions, source hashes, timing, and `summary.json` to Drive.

Mount Drive, confirm setup and smoke passed, then set `RUN_BASELINE=True`. Use a new output folder for every run. The defaults read persistent Drive data; optionally set the data roots to an existing local cache for faster loading. No training or attention changes are made. PAMR failures are recorded and invalidate a clean baseline.

While this cell runs, prints appear directly below it. A single notebook kernel executes one cell at a time. To inspect files while it is busy, use the extension's Colab Terminal or another notebook kernel; the result cell below can run after this one finishes.


In [ ]:
from datetime import datetime

RUN_BASELINE = False
DATASET = 'ade20k'  # 'ade20k' or 'pc59'
STAGE = 'subset'   # 'subset' or 'full'
SUBSET_SIZE = 32
SEED = 42
RUN_TAG = 'extension-' + datetime.now().strftime('%Y%m%d-%H%M%S')
DATA_ROOTS = {
    'ade20k': '/content/drive/MyDrive/datasets/ADEChallengeData2016',
    'pc59': '/content/drive/MyDrive/datasets/VOCdevkit/VOC2010',
}
if DATASET not in DATA_ROOTS or STAGE not in {'subset', 'full'}:
    raise ValueError('Choose a supported DATASET and STAGE.')
BASELINE_WORK_DIR = Path('/content/drive/MyDrive/NACLIP-results') / RUN_TAG / f'{DATASET}_{STAGE}'
if RUN_BASELINE:
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Mount Google Drive first.')
    if not PYTHON.is_file():
        raise RuntimeError('Run isolated dependency setup first.')
    print('Saving results to:', BASELINE_WORK_DIR, flush=True)
    run_remote([str(PYTHON), '-u', 'tools/run_baseline.py',
                '--dataset', DATASET, '--stage', STAGE,
                '--data-root', DATA_ROOTS[DATASET],
                '--work-dir', str(BASELINE_WORK_DIR),
                '--subset-size', str(SUBSET_SIZE), '--seed', str(SEED)], cwd=PROJECT)
else:
    print('Baseline disabled. Choose dataset/stage, then set RUN_BASELINE=True.')


## Inspect progress and results

`RESULT_DIR` defaults to the preceding baseline cell's output folder; set it to an existing CLI or extension run to inspect that run instead. `progress.json` updates every 25 images. `summary.json` contains final mIoU, timing and `baseline_valid`; it is written only after evaluation completes. Check `pamr_failures` before calling a run a clean reproduction. The reference README reports ADE20K 19.05 and PC59 38.35 mIoU; subset scores are not benchmark results.


In [ ]:
import json
from IPython.display import display, Image as DisplayImage

RESULT_DIR = BASELINE_WORK_DIR  # Or Path('/content/drive/MyDrive/NACLIP-results/<run>/<dataset>_<stage>')
for filename in ['progress.json', 'summary.json']:
    path = RESULT_DIR / filename
    if path.is_file():
        print(filename)
        print(json.dumps(json.loads(path.read_text()), indent=2))
    else:
        print(f'{filename}: not present in {RESULT_DIR}')
comparisons = sorted((RESULT_DIR / 'comparisons').glob('*.jpg'))
if comparisons:
    print('Example: input / ground truth overlay / prediction overlay')
    display(DisplayImage(filename=str(comparisons[0])))
